<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 El Cuaderno de la Tienda: Usar los Datos con Respeto 🔐
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 04
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/04%20-%20No%20Parametrica%20Regresion%20y%20Etica/Para%20Dummies/03_Etica_Datos_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no matemáticos»** del [cuaderno 03 del Módulo 04 (Ética en el uso de datos)](../03_Etica_en_el_Uso_de_Datos.ipynb). Los datos casi siempre son **de personas**: lo que compraron, dónde viven, qué enfermedad tienen. Quien los maneja tiene una responsabilidad: **proteger** a la gente, **no engañarse** (ni engañar) con los resultados y **tratar a todos con el mismo rasero**. Usamos la libreta de una **tienda de vereda** (todo **inventado**).

| Lo que decimos | Lo que significa en la vida diaria |
|---|---|
| **Anonimizar** | Que nadie pueda saber de quién es cada fila, **ni cruzando** con otras listas |
| **Combinación única** | Una mezcla de datos (vereda + edad + oficio) que solo cumple **una** persona |
| **Ruido que protege** | Mover un poquito la respuesta para que nadie pueda reconstruir un dato individual |
| **Buscar y buscar** | Probar muchas hipótesis hasta que alguna «salga», que por pura suerte casi siempre ocurre |
| **Mismo rasero** | Que una decisión (un crédito, por ejemplo) no castigue a un grupo por lo que no tiene que ver con su capacidad |

Al terminar podrás explicar:
1. Por qué **quitar el nombre no basta** y cómo **proteger** con grupos más grandes y con ruido.
2. Por qué **probar 20 cosas** casi siempre encuentra «algo» y cómo evitar engañarse.
3. Por qué un computador que **aprende del pasado** puede **repetir** las injusticias del pasado.

> 🔗 Si luego quieres ver la versión **con k-anonimato, privacidad diferencial, Bonferroni, Benjamini-Hochberg y métricas de equidad**: [cuaderno estándar](../03_Etica_en_el_Uso_de_Datos.ipynb).

---
## 1. Dos Escudos para Proteger a las Personas 🛡️

La tienda guarda **400 clientes** sin nombre: **vereda**, **edad**, **oficio** y si compraron **remedio para la presión** (un dato **sensible**). ¿Está protegido? Veamos cuántas personas son **únicas** por su combinación de datos, y cómo se reducen los riesgos **agrupando** las edades. Después probamos el segundo escudo: **responder con un poquito de ruido**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)                      # semilla fija: siempre salen los mismos numeros
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})

n = 400
tienda = pd.DataFrame({
    "vereda": rng.choice(["Norte", "Sur", "Centro", "Alto", "Bajo", "Llano"], n, p=[0.25, 0.2, 0.2, 0.15, 0.1, 0.1]),
    "edad": rng.integers(18, 80, n),
    "oficio": rng.choice(["agricultor", "tendero", "docente", "transportador", "otro"], n, p=[0.45, 0.15, 0.1, 0.15, 0.15]),
})
tienda["remedio_presion"] = rng.random(n) < (0.02 + 0.006 * (tienda["edad"] - 18))      # dato sensible (inventado)

def personas_unicas(df, columnas):
    """Cuantas personas son las UNICAS con su combinacion de datos."""
    tamano_grupo = df.groupby(columnas)[columnas[0]].transform("size")
    return int((tamano_grupo == 1).sum()), int(df.groupby(columnas).size().min())

u1, minimo1 = personas_unicas(tienda, ["vereda", "oficio"])
u2, minimo2 = personas_unicas(tienda, ["vereda", "edad", "oficio"])
tienda["edad_grupo"] = (tienda["edad"] // 20 * 20).astype(str) + "-" + (tienda["edad"] // 20 * 20 + 19).astype(str)       # edades en grupos de 20 anos
u3, minimo3 = personas_unicas(tienda, ["vereda", "edad_grupo", "oficio"])
print(f"vereda + oficio              : {u1:3d} personas unicas | el grupo mas chico tiene {minimo1} personas")
print(f"vereda + EDAD EXACTA + oficio: {u2:3d} personas unicas | el grupo mas chico tiene {minimo2} persona")
print(f"vereda + edad en grupos + oficio: {u3:3d} personas unicas | el grupo mas chico tiene {minimo3} persona(s)")

# Alguien que conozca a un cliente (vereda, edad exacta y oficio) puede descubrir su dato sensible
unica = tienda[tienda.groupby(["vereda", "edad", "oficio"])["vereda"].transform("size") == 1].iloc[0]
print(f"\nUn vecino que sabe que Don X vive en {unica.vereda}, tiene {unica.edad} anos y es {unica.oficio} encuentra UNA sola fila: compro remedio para la presion = {bool(unica.remedio_presion)}")

# Segundo escudo: responder con ruido
real = int(((tienda.vereda == "Alto") & tienda.remedio_presion).sum())
una_vez = real + rng.laplace(0, 2)                                                    # una respuesta con ruido
mil_veces = (real + rng.laplace(0, 2, 1000)).mean()                                   # el que pregunta 1 000 veces y promedia
print(f"\nCuantos clientes de la vereda Alto compraron el remedio? Verdad = {real} | una respuesta con ruido = {una_vez:.1f} | promedio de 1 000 respuestas con ruido = {mil_veces:.2f}")

assert u2 > 200 and u3 < u2 / 10
assert minimo2 == 1 and minimo3 >= 1 and abs(mil_veces - real) < 0.4

---
**Lección:** con **vereda + oficio** nadie es único (el grupo más pequeño tiene 2 personas), pero al añadir la **edad exacta**, **307 de los 400 clientes (77 %)** quedan como únicos: cualquiera que conozca la vereda, la edad y el oficio de un vecino encuentra **una sola fila** y descubre su dato sensible. Agrupando las edades de 20 en 20 años los únicos bajan a **18** (4.5 %): mucho mejor, aunque no perfecto, y se pierde detalle (ya no sabes la edad exacta). El segundo escudo es el **ruido**: la respuesta con ruido (16.8) oculta el número verdadero (17), pero quien pregunta **mil veces lo mismo** y promedia obtiene **17.07**: cada pregunta gasta un poco de protección, por eso se lleva la cuenta y se limita cuántas veces se puede preguntar.

---
## 2. Probar Veinte Colores de Camisa 👕

Un investigador quiere saber si **el color de la camisa** tiene que ver con **ganar la lotería**. Revisa **20 colores**, cada uno contra el resultado de **100 personas**. Aquí **no hay ninguna relación real** (los datos son pura suerte). ¿Cuántas veces encuentra algún color «significativo»? Decimos que un color sale **«raro»** si su diferencia es tan grande que el azar la produce menos de 5 veces de cada 100.

In [ ]:
def experimento(n_colores, umbral, repeticiones=2000, personas=100, semilla=0):
    """Fraccion de experimentos en que ALGUN color sale 'raro' cuando en realidad no hay ninguna relacion."""
    g = np.random.default_rng(semilla)
    resultado = g.normal(size=(repeticiones, 1, personas))                                 # ganancia de cada persona (pura suerte)
    lleva = g.random((repeticiones, n_colores, personas)) < 0.5                            # lleva o no lleva ese color de camisa
    n1 = lleva.sum(2); n0 = personas - n1
    m1 = (resultado * lleva).sum(2) / n1; m0 = (resultado * ~lleva).sum(2) / n0
    s1 = np.sqrt(((resultado - m1[..., None]) ** 2 * lleva).sum(2) / (n1 - 1)); s0 = np.sqrt(((resultado - m0[..., None]) ** 2 * ~lleva).sum(2) / (n0 - 1))
    z = (m1 - m0) / np.sqrt(s1 ** 2 / n1 + s0 ** 2 / n0)
    return float((np.abs(z) > umbral).any(axis=1).mean())

un_color = experimento(1, 1.96)
veinte = experimento(20, 1.96)
veinte_estricto = experimento(20, 3.02)                                                   # exigir mucho mas: dividir el 5 % entre los 20 intentos
print(f"Probando 1 solo color        : sale 'raro' en {un_color:.1%} de los experimentos (lo esperado: 5 %)")
print(f"Probando 20 colores          : ALGUN color sale 'raro' en {veinte:.1%} (si fueran independientes: 1 - 0.95^20 = {1 - 0.95 ** 20:.1%})")
print(f"20 colores, exigiendo mas    : baja a {veinte_estricto:.1%}")

plt.figure(figsize=(8, 3.6))
plt.bar(["1 color", "20 colores", "20 colores\nexigiendo mas"], [un_color, veinte, veinte_estricto], color=["#16a34a", "#dc2626", "#2563eb"])
plt.axhline(0.05, color="black", ls="--", label="5 %"); plt.ylabel("experimentos con un 'hallazgo' falso"); plt.title("Sin relacion real, buscar mucho encuentra algo"); plt.legend(); plt.tight_layout(); plt.show()

assert abs(un_color - 0.05) < 0.02 and abs(veinte - (1 - 0.95 ** 20)) < 0.04 and veinte_estricto < 0.09

---
##### 🛠️ Práctica 1: Cuarenta colores: ¿qué tan seguro es el «hallazgo»?

Con la función `experimento(n_colores, umbral)` de la sección 2:

1. Calcula `con_40 = experimento(40, 1.96)`: la fracción de experimentos que «encuentran» algún color raro entre **40** colores (sin relación real).
2. Calcula `esperado_40 = 1 - 0.95 ** 40`.
3. Calcula `con_40_estricto = experimento(40, 3.23)`, exigiendo más (el 3.23 equivale a dividir el 5 % entre los 40 intentos).
4. Comprueba con `assert` que `con_40` está a menos de 0.04 de `esperado_40` y que `con_40_estricto < 0.09`. Comenta: si un artículo dice «de 40 variables, 2 salieron significativas», ¿qué te preguntarías?

In [ ]:
# Escribe tu código aquí

# con_40 = ...
# esperado_40 = ...
# con_40_estricto = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
con_40 = experimento(40, 1.96)
esperado_40 = 1 - 0.95 ** 40
con_40_estricto = experimento(40, 3.23)
print(f"Con 40 colores: algun 'hallazgo' falso en {con_40:.1%} (esperado {esperado_40:.1%}) | exigiendo mas: {con_40_estricto:.1%}")

assert abs(con_40 - esperado_40) < 0.04 and con_40_estricto < 0.09
print("Con 40 intentos es casi seguro que algo salga 'raro' por pura suerte. Hay que preguntar: ¿cuantas cosas se probaron en total?, ¿estaba planeado?, ¿se repite con datos nuevos?")
```
</details>

---

---
**Lección:** con **un solo color**, el azar produce un «hallazgo» en **5.0 %** de los experimentos, como debe ser. Con **20 colores**, en el **65.3 %** de los experimentos *algún* color sale «raro» aunque no haya ninguna relación (la cuenta teórica da 64.2 %). Con 40 colores (Práctica 1), **89.0 %**. Si en vez de aceptar cualquier cosa «rara» se **exige mucho más** (dividir el 5 % entre los intentos), el error baja a **6.3 %** con 20 colores y a **6.8 %** con 40: casi vuelve al 5 % que queríamos. Moraleja: no basta con que «algo salió»; hay que saber **cuántas cosas se probaron** y confirmar con datos nuevos.

---
## 3. El Mismo Rasero para Todos ⚖️

Una cooperativa decide **a quién prestar** para sembrar. Cada solicitante tiene un **puntaje** de capacidad de pago (inventado; el grupo B tiene un puntaje promedio un poco menor por razones de ingresos, **pero a igual puntaje paga igual**). En el pasado, los funcionarios fueron **más duros con el grupo B** de lo que su capacidad justificaba. Si el computador **copia el pasado**, ¿qué pasa? Miramos, de las personas que **sí habrían pagado**, a cuántas se les aprueba en cada grupo.

In [ ]:
g = np.random.default_rng(21)
m = 4000
grupo = np.where(g.random(m) < 0.4, "B", "A")
puntaje = g.normal(np.where(grupo == "B", -0.25, 0.0), 1.0)                       # capacidad de pago (B algo menor en promedio)
paga = (puntaje + g.normal(0, 0.7, m)) > 0                                         # quien de verdad pagaria
aprob_pasado = (puntaje - np.where(grupo == "B", 0.8, 0.0)) > 0                    # el pasado: mismo puntaje, pero mas exigente con B
aprob_justa = puntaje > 0                                                          # un solo rasero: el mismo umbral para todos

def resumen(aprueba):
    filas = {}
    for gr in ("A", "B"):
        sel = grupo == gr
        filas[f"grupo {gr}"] = {"se aprueba al": aprueba[sel].mean(), "de los que SI pagarian, se aprueba al": aprueba[sel & paga].mean()}
    return pd.DataFrame(filas).T

print("COPIANDO EL PASADO:"); print((100 * resumen(aprob_pasado)).round(0).astype(int).astype(str).add(" %").to_string())
print("\nUN SOLO RASERO (mismo umbral para todos):"); print((100 * resumen(aprob_justa)).round(0).astype(int).astype(str).add(" %").to_string())

r_pasado, r_justa = resumen(aprob_pasado), resumen(aprob_justa)
brecha_pasado = r_pasado.iloc[0, 1] - r_pasado.iloc[1, 1]; brecha_justa = r_justa.iloc[0, 1] - r_justa.iloc[1, 1]
print(f"\nDiferencia entre grupos en 'de los que SI pagarian, se aprueba al': copiando el pasado = {100 * brecha_pasado:.0f} puntos | un solo rasero = {100 * brecha_justa:.0f} puntos")

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.bar(np.arange(2) - 0.2, r_pasado.iloc[:, 1], 0.4, color="#dc2626", label="copiando el pasado"); ax.bar(np.arange(2) + 0.2, r_justa.iloc[:, 1], 0.4, color="#16a34a", label="un solo rasero")
ax.set_xticks(range(2)); ax.set_xticklabels(["grupo A", "grupo B"]); ax.set_ylabel("aprobados entre quienes SI pagarian"); ax.set_title("Copiar el pasado castiga al grupo B"); ax.legend(); plt.tight_layout(); plt.show()

assert brecha_pasado > 2 * brecha_justa and brecha_pasado > 0.15

---
**Lección:** si el computador **copia el pasado**, aprueba al **50 %** del grupo A y solo al **16 %** del B; y de las personas que **sí habrían pagado**, aprueba al **80 %** en A y apenas al **36 %** en B: una brecha de **44 puntos** que no tiene que ver con la capacidad de pago. Con **un solo rasero** (el mismo umbral para todos) la brecha baja a **7 puntos** (80 % y 74 %). Ojo: no llega a cero, porque el grupo B tiene un puntaje promedio algo menor por razones de ingresos: ningún ajuste técnico borra las diferencias de fondo, y por eso hay que **discutirlas** y no solo calcularlas. Además, un programa que «no mira el grupo» puede igual repetir el sesgo si copia el pasado o usa datos que lo delatan (la vereda, el barrio...).

---
## Resumen en una frase 🎯

> **Quitar el nombre no anonimiza (las combinaciones de datos señalan a una persona), por eso se agrupa y se responde con un poquito de ruido; probar muchas cosas hasta que alguna salga «rara» engaña casi siempre, y un computador que aprende de decisiones pasadas puede repetir sus injusticias, así que hay que revisar los resultados grupo por grupo.**

### Lo que aprendiste hoy:
- ✅ **Combinaciones únicas:** vereda + edad exacta + oficio señalan a una persona; **agrupar** las edades baja el riesgo (a costa de detalle).
- ✅ **Ruido que protege:** una respuesta con ruido esconde a la persona, pero **preguntar mil veces** y promediar lo deshace: cada pregunta gasta protección.
- ✅ **Buscar hasta encontrar:** con 20 intentos, casi siempre aparece algo «raro» por pura suerte; se exige más (o se planea antes de mirar).
- ✅ **Mismo rasero:** copiar decisiones del pasado puede castigar a un grupo; se revisa **a cuántos de los que sí pagarían** se aprueba en cada grupo.

### 🧠 ¿Ya lo tienes claro?

1. Te piden publicar una tabla de pacientes «sin nombres» pero con barrio, edad y profesión. ¿Qué le preguntarías a quien la pidió y qué le propondrías para proteger a las personas?
2. Una amiga prueba 30 dietas y encuentra que una «funciona» con un resultado raro. ¿Qué le dirías antes de que la recomiende a otras personas?
3. Un banco dice: «nuestro programa nunca mira el género». ¿Eso asegura que sea justo? ¿Qué revisarías?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos (Edición Didáctica Para Dummies)</i>
  </p>
</div>